# **EDA Notebook**



---
## 0. Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 68.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [2]:
# <Student to fill this section>
group_name = "Neural Mind"
student_name = "Aryaman Malik"
student_id = "25980501"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# <Student to fill this section>

### 0.b Import Packages

In [6]:
# <Student to fill this section>
import pandas as pd
import altair as alt

---
## B. Data Understanding

In [7]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "product_sub_category.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [8]:
# <Student to fill this section>
# Shape, schema, head
print("Shape:", df.shape)
df.info()
df.head()

Shape: (37, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   product_subcategory_id  37 non-null     object
 1   product_category_id     37 non-null     object
 2   name                    37 non-null     object
dtypes: object(3)
memory usage: 1020.0+ bytes


,product_subcategory_id,product_category_id,name
0,5332b46f-2ec0-453b-8237-c78051d4c7bf,6b440a50-1a1b-4d3b-988f-076c498bd092,Mountain Bikes
1,5fb7ef8d-7610-4230-ae2a-09846fd56ada,6b440a50-1a1b-4d3b-988f-076c498bd092,Road Bikes
2,be3c8ce6-c1ec-4835-af3b-e49e199380c5,6b440a50-1a1b-4d3b-988f-076c498bd092,Touring Bikes
3,327e57d0-2de2-4b3f-8eff-cb6b8a273842,65e2841b-c7da-4c4a-9565-cd8cfd9a2f68,Handlebars
4,57591aab-ca01-4a76-9c74-e7a806b42ba5,65e2841b-c7da-4c4a-9565-cd8cfd9a2f68,Bottom Brackets


In [9]:
# Quality checks
print("Missing values:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Duplicate subcategory_id: {df['product_subcategory_id'].duplicated().sum()}")
print(f"Duplicate names: {df['name'].duplicated().sum()}")

print("\nUnique counts:")
for c in df.columns:
    print(f"  {c}: {df[c].nunique()}")

print("\nSubcategories per top-level category (counts):")
print(df['product_category_id'].value_counts())

Missing values:
product_subcategory_id    0
product_category_id       0
name                      0
dtype: int64

Duplicate rows: 0
Duplicate subcategory_id: 0
Duplicate names: 0

Unique counts:
  product_subcategory_id: 37
  product_category_id: 4
  name: 37

Subcategories per top-level category (counts):
product_category_id
65e2841b-c7da-4c4a-9565-cd8cfd9a2f68    14
0a37a9c4-764e-4952-bf4f-d47b38694c4d    12
2664e8d8-c7ce-4a38-aae4-07e0b536bba3     8
6b440a50-1a1b-4d3b-988f-076c498bd092     3
Name: count, dtype: int64


In [10]:
# <Student to fill this section>
dataset_insights = """
product_sub_category is a small dimension table with 37 rows and 3 columns: product_subcategory_id, product_category_id and name. It defines the mid-tier of a 3-level product hierarchy (category > subcategory > product).

Data quality is excellent:
- Zero missing values, zero duplicates.
- 37 unique subcategory_ids and 37 unique subcategory names (no name collisions).
- All 37 subcategories map to exactly one of 4 parent product_category_ids, with the parent distribution: 14 / 12 / 8 / 3.

Why this matters for the anomaly use case:
The subcategory is the natural unit for computing baselines. Pricing, discounting and quantity behaviour are very different across "Road Bikes" (high price, low volume) versus "Tires and Tubes" (low price, high volume). The anomaly model will use product_subcategory_id to group lines and compute subcategory-specific medians (median price, median discount, median quantity) so that what is normal in one subcategory does not get flagged as anomalous when it appears in another. This is more meaningful than the 4-level top category and more stable than working with 500+ individual products.
"""

In [11]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `\<product_category_id\>`

In [12]:
# <Student to fill this section>
# Join to product_category to get the parent category names
cat_df = pd.read_csv(at.folder_path / "product_category.csv").rename(columns={'name':'category_name'})
sub_named = df.rename(columns={'name':'subcategory_name'}).merge(cat_df, on='product_category_id', how='left')

print("Subcategories belonging to each top-level category:")
print(sub_named.groupby('category_name').size().sort_values(ascending=False))

print("\nFull mapping (subcategory -> category):")
print(sub_named[['category_name','subcategory_name']]
        .sort_values(['category_name','subcategory_name']).to_string(index=False))

Subcategories belonging to each top-level category:
category_name
Components     14
Accessories    12
Clothing        8
Bikes           3
dtype: int64

Full mapping (subcategory -> category):
category_name  subcategory_name
  Accessories        Bike Racks
  Accessories       Bike Stands
  Accessories Bottles and Cages
  Accessories          Cleaners
  Accessories           Fenders
  Accessories           Helmets
  Accessories   Hydration Packs
  Accessories            Lights
  Accessories             Locks
  Accessories          Panniers
  Accessories             Pumps
  Accessories   Tires and Tubes
        Bikes    Mountain Bikes
        Bikes        Road Bikes
        Bikes     Touring Bikes
     Clothing        Bib-Shorts
     Clothing              Caps
     Clothing            Gloves
     Clothing           Jerseys
     Clothing            Shorts
     Clothing             Socks
     Clothing            Tights
     Clothing             Vests
   Components   Bottom Brackets
   Compo

In [13]:
# Subcategories per top-level category
chart = alt.Chart(sub_named).mark_bar(color='#4C78A8').encode(
    x=alt.X('category_name:N', sort='-y', title='Top-level category'),
    y=alt.Y('count():Q', title='Number of subcategories'),
    tooltip=['category_name','count():Q']
).properties(
    title='Number of subcategories within each top-level category',
    width=450, height=300
)
chart

alt.Chart(...)

In [14]:
# <Student to fill this section>
feature_1_insights = """
product_category_id is the foreign key that links each subcategory to one of the 4 top-level categories. The distribution is uneven by design:
- Components: 14 subcategories (brakes, chains, cranksets, derailleurs, forks, handlebars, headsets, mountain frames, pedals, road frames, saddles, touring frames, wheels, bottom brackets) - the largest group because every bike has many discrete parts.
- Accessories: 12 subcategories (helmets, lights, locks, pumps, panniers, etc.) - peripheral items sold to riders.
- Clothing: 8 subcategories (jerseys, shorts, gloves, etc.).
- Bikes: only 3 subcategories (Mountain Bikes, Road Bikes, Touring Bikes) - but these are the highest-value items.

Implications:
- The Bikes category is small in subcategory count but accounts for almost all of the high-value sales (median price >USD 1,000) - the anomaly model must learn to treat a Bikes line very differently from a Clothing or Accessories line.
- Top-level category is a strong, low-cardinality feature (4 levels) that can be safely one-hot encoded in the Preparation notebook. Subcategory (37 levels) will need either ordinal/target encoding or aggregation depending on the algorithm we choose.
"""

In [15]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `\<name and product_distribution\>`

In [16]:
# <Student to fill this section>
# How many priced products live under each subcategory?
prod_df = pd.read_csv(at.folder_path / "product.csv").drop_duplicates('product_id')
prod_df = prod_df.rename(columns={'name':'product_name'})

joined = prod_df.merge(
    df.rename(columns={'name':'subcategory_name'}),
    on='product_subcategory_id', how='left'
)
sub_counts = (joined['subcategory_name'].value_counts(dropna=True)
                .rename_axis('subcategory').reset_index(name='product_count'))
print("Products per subcategory (top 15):")
print(sub_counts.head(15))
print(f"\nTotal subcategories with at least 1 product: {(sub_counts['product_count']>0).sum()} / 37")
print(f"Subcategories with only 1 product: {(sub_counts['product_count']==1).sum()}")

Products per subcategory (top 15):
        subcategory  product_count
0        Road Bikes             43
1       Road Frames             33
2    Mountain Bikes             32
3   Mountain Frames             28
4     Touring Bikes             22
5    Touring Frames             18
6            Wheels             14
7   Tires and Tubes             11
8           Saddles              9
9        Handlebars              8
10          Jerseys              8
11           Pedals              7
12           Shorts              7
13           Gloves              6
14            Socks              4

Total subcategories with at least 1 product: 37 / 37
Subcategories with only 1 product: 9


In [17]:
# Bar chart of products per subcategory
chart = alt.Chart(sub_counts).mark_bar(color='#54A24B').encode(
    x=alt.X('product_count:Q', title='Number of products'),
    y=alt.Y('subcategory:N', sort='-x', title='Subcategory'),
    tooltip=['subcategory','product_count']
).properties(
    title='Products per subcategory (deduplicated)',
    width=500, height=550
)
chart

alt.Chart(...)

In [18]:
feature_2_insights = """
The subcategory name field is a clean, unique label (37 distinct values, no nulls, no duplicates) but the population of products under each subcategory is highly imbalanced:
- The top 6 subcategories (Road Bikes 43, Road Frames 33, Mountain Bikes 32, Mountain Frames 28, Touring Bikes 22, Touring Frames 18) account for 176 of the 295 priced products (~60%).
- 9 subcategories have only 1 product each. Subcategory-level statistics for these will be unreliable (a "median" computed on a single observation is meaningless).

Implications for modelling:
- For subcategory-level baseline features (median price, median discount, median quantity), we will fall back to the parent category median when the subcategory has fewer than ~5 products. This keeps the baseline stable.
- The skew also affects how we interpret anomaly scores: a flagged anomaly inside Road Bikes (43 products of context) is much more reliable than one inside a single-product subcategory (no real distribution to compare to).
- We will retain the full subcategory granularity for the join but engineer features at both subcategory AND parent-category level to give the model both fine and coarse context.
"""


In [19]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `\<price_profile_per_subcategory\>`

In [20]:
# <Student to fill this section>
# Price profile per subcategory using only priced products
priced = joined[joined['list_price'] > 0]
sub_price = (priced.groupby('subcategory_name')
                .agg(n_products=('product_id','nunique'),
                     min_price=('list_price','min'),
                     median_price=('list_price','median'),
                     max_price=('list_price','max'))
                .sort_values('median_price', ascending=False)
                .reset_index())
print("Top 10 most expensive subcategories (by median price):")
print(sub_price.head(10))
print("\nBottom 10 cheapest subcategories:")
print(sub_price.tail(10))

print(f"\nSpread: cheapest median (${sub_price['median_price'].min():.2f}) "
      f"vs most expensive median (${sub_price['median_price'].max():.2f}) "
      f"= {sub_price['median_price'].max()/sub_price['median_price'].min():.0f}x ratio")

Top 10 most expensive subcategories (by median price):
  subcategory_name  n_products  min_price  median_price  max_price
0       Road Bikes          43    539.990     1457.9900    3578.27
1    Touring Bikes          22    742.350     1214.8500    2384.07
2   Mountain Bikes          32    539.990     1079.9900    3399.99
3      Road Frames          33    337.220      594.8300    1431.50
4  Mountain Frames          28    249.790      356.4250    1364.50
5   Touring Frames          18    333.420      333.4200    1003.91
6        Cranksets           3    175.490      256.4900     404.99
7           Wheels          14     60.745      240.5175     357.06
8            Forks           3    148.220      175.4900     229.49
9      Bike Stands           1    159.000      159.0000     159.00

Bottom 10 cheapest subcategories:
     subcategory_name  n_products  min_price  median_price  max_price
27             Gloves           6      24.49        31.240      37.99
28              Locks           1

In [21]:
import altair as alt

# Defensive: ensure no zeros/nulls would break a log scale, and reset index
plot_df = sub_price.copy()
plot_df = plot_df[plot_df['median_price'] > 0]

chart = alt.Chart(plot_df).mark_bar(color='#F58518').encode(
    x=alt.X('median_price:Q', title='Median list price (USD)'),
    y=alt.Y('subcategory_name:N', sort='-x', title='Subcategory'),
    tooltip=['subcategory_name', 'n_products', 'median_price', 'min_price', 'max_price']
).properties(
    title='Median list price by subcategory',
    width=600,
    height=550
)

chart.display()   # force-render even if the cell has subsequent code


alt.Chart(...)

In [22]:
# <Student to fill this section>
feature_n_insights = """
Subcategory is by far the strongest single predictor of "normal" price for a sales line. The median list price varies by roughly 183x across subcategories:
- The 3 Bike subcategories dominate the high end: Road Bikes (median USD 1,458), Touring Bikes (USD 1,215), Mountain Bikes (USD 1,080).
- A second tier of frame subcategories sits between USD 333 and USD 595.
- A long tail of accessories and clothing items sit below USD 100, with Cleaners (USD 7.95), Caps (USD 8.99) and Bottles and Cages (USD 8.99) at the bottom.

This is the single most important insight for the anomaly model:
A line item with unit_price = USD 50 is completely normal in the Saddles or Jerseys subcategory, but it would be a major anomaly in the Road Bikes subcategory (where the cheapest product is USD 540). Without conditioning on subcategory, an Isolation Forest would learn a single "global" notion of normal price that fails on both ends of the spectrum.

We will therefore engineer ratio-based features in the Preparation notebook:
- price_ratio = unit_price / subcategory_median_list_price
- discount_ratio = unit_price_discount / subcategory_median_discount
These ratios will be much more informative than the raw monetary values and will make the model robust to the heavy-tailed cross-subcategory price distribution.
"""

In [23]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)